# 🗂️ Notebook 2: Data Model, Code Storage & APIs

In this notebook we build the **control plane** — the part that accepts code uploads,
stores configs, and answers the question *"given a function name, what should I run?"*.

We'll build it **three times**, each version fixing a real problem:

1. ❌ **Naive dict store** — no validation, no versioning, stringly-typed.
2. ⚠️ **Validated with pydantic** — types catch bugs at the boundary.
3. ✅ **Production-ish** — immutable versions + read-through metadata cache.

## 🛠️ Setup

```bash
cd 06-system-designs/amazon-lambda
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.

> 💡 This lab has **no Docker / no cloud dependencies**. Everything is simulated in pure Python
> so you can see the moving parts clearly. Real Lambda uses Firecracker microVMs, S3, DynamoDB,
> and SQS under the hood — we'll point out where those fit as we go.


## ❌ Approach 1: Naive dict store

Let's start ugly on purpose. Everything in one dict, no validation.

In [ ]:
# A "database" that is just a dict.
functions = {}

def create_function_bad(cfg):
    functions[cfg["name"]] = cfg

def invoke_function_bad(name, event):
    cfg = functions[name]   # KeyError if missing, crashes
    # pretend we run the code
    return f"ran {cfg['name']} with {event}"

# Register something
create_function_bad({"name": "resize", "runtime": "python3.11", "memory": 256})
print(invoke_function_bad("resize", {"url": "cat.jpg"}))

### What's wrong?

| Problem | Consequence |
|---|---|
| `memory` can be a string, negative, 1 PB — nothing checks. | Worker crashes at runtime. |
| Missing `handler`, `timeout`, `code_url` — not caught. | Silent failures in prod. |
| Overwriting a function is silent — no versioning. | Can't roll back a bad deploy. |
| Unknown function = `KeyError`. | Ugly 500 instead of a clean 404. |

Let's fix the validation part first.

## ⚠️ Approach 2: Pydantic models

[pydantic](https://docs.pydantic.dev/) checks types at the edge so the rest of your code doesn't have to.
Think of it as a bouncer at the door of your service.

In [ ]:
from datetime import datetime
from typing import Literal, Optional
from pydantic import BaseModel, Field, ValidationError

Runtime = Literal["python3.11", "node20", "go1.21"]
Status  = Literal["ok", "error", "timeout"]

class FunctionConfig(BaseModel):
    name: str = Field(min_length=1, max_length=64)
    runtime: Runtime
    handler: str                      # e.g. "main.handler"
    memory_mb: int = Field(ge=128, le=10_240)   # AWS limits
    timeout_s: int = Field(ge=1, le=900, default=30)
    code_url: str                     # s3://bucket/key.zip

class Invocation(BaseModel):
    id: str
    function: str
    started_at: datetime
    duration_ms: Optional[int] = None
    cold_start: bool = False
    status: Status = "ok"

# Good config: validates fine.
cfg = FunctionConfig(
    name="resize", runtime="python3.11", handler="main.handler",
    memory_mb=256, code_url="s3://my-code/resize/v1.zip",
)
print("OK:", cfg)

# Bad config: memory too big, bad runtime.
try:
    FunctionConfig(name="x", runtime="cobol", handler="h",
                   memory_mb=999_999, code_url="s3://x/y")
except ValidationError as e:
    print("\nREJECTED:")
    print(e)

Validation at the boundary means the *rest* of the system can assume inputs are sane. Huge win.

## ✅ Approach 3: Versioned store + metadata cache

Real Lambda deployments are **immutable** — each deploy creates a new version.
This lets you roll back instantly by pointing an *alias* (e.g. `prod`) at a prior version.

Also: the invoker **must not** hit the metadata DB on every call at 600 k QPS.
We add a simple **TTL cache** to simulate what the real front-end does in memory.

In [ ]:
import hashlib, time

class FakeS3:
    """Pretend S3: content-addressable code storage."""
    def __init__(self): self.blobs = {}
    def put(self, zip_bytes: bytes) -> str:
        key = hashlib.sha256(zip_bytes).hexdigest()[:12]
        self.blobs[key] = zip_bytes
        return f"s3://code/{key}.zip"
    def get(self, url: str) -> bytes:
        return self.blobs[url.split("/")[-1].removesuffix(".zip")]

class FunctionStore:
    """Versioned metadata store. All versions are immutable."""
    def __init__(self):
        self._versions = {}   # (name, version) -> FunctionConfig
        self._aliases  = {}   # (name, alias)   -> version
        self._latest   = {}   # name            -> int

    def publish(self, cfg: FunctionConfig) -> int:
        v = self._latest.get(cfg.name, 0) + 1
        self._versions[(cfg.name, v)] = cfg
        self._latest[cfg.name] = v
        return v

    def set_alias(self, name: str, alias: str, version: int):
        assert (name, version) in self._versions, "version does not exist"
        self._aliases[(name, alias)] = version

    def resolve(self, name: str, alias_or_version="prod"):
        if isinstance(alias_or_version, int):
            return self._versions[(name, alias_or_version)], alias_or_version
        v = self._aliases[(name, alias_or_version)]
        return self._versions[(name, v)], v


class MetadataCache:
    """Tiny TTL cache. Real invokers use one like this to avoid hitting DynamoDB."""
    def __init__(self, store: FunctionStore, ttl_s: float = 60.0):
        self.store = store
        self.ttl = ttl_s
        self._cache = {}        # key -> (cfg, version, expires_at)
        self.hits = self.misses = 0

    def resolve(self, name, alias="prod"):
        key = (name, alias)
        entry = self._cache.get(key)
        now = time.time()
        if entry and entry[2] > now:
            self.hits += 1
            return entry[:2]
        self.misses += 1
        cfg, v = self.store.resolve(name, alias)
        self._cache[key] = (cfg, v, now + self.ttl)
        return cfg, v


# ── Try it out ─────────────────────────────────────────────
s3    = FakeS3()
store = FunctionStore()
cache = MetadataCache(store, ttl_s=60)

code_url = s3.put(b"def handler(e): return {'ok': True, 'echo': e}")
v1 = store.publish(FunctionConfig(
    name="echo", runtime="python3.11", handler="main.handler",
    memory_mb=128, code_url=code_url,
))
store.set_alias("echo", "prod", v1)

# Publish a new version + flip the alias (= deploy).
code_url2 = s3.put(b"def handler(e): return {'ok': True, 'echo_v2': e}")
v2 = store.publish(FunctionConfig(
    name="echo", runtime="python3.11", handler="main.handler",
    memory_mb=128, code_url=code_url2,
))
store.set_alias("echo", "prod", v2)
print(f"Published v{v1}, v{v2}. Alias 'prod' -> v{v2}")

# Rollback is one line:
store.set_alias("echo", "prod", v1)
print("Rolled back: alias 'prod' now ->", store._aliases[("echo", "prod")])

# Cache behavior:
for _ in range(1000): cache.resolve("echo", "prod")
print(f"Cache: {cache.hits} hits, {cache.misses} misses (hit rate {cache.hits/(cache.hits+cache.misses):.1%})")

### Why this matters

- **Immutable versions** make rollback atomic and safe (just flip the alias pointer).
- **Alias indirection** is how blue/green and canary deploys work: point 1% of traffic to `v2`, rest to `v1`.
- The **metadata cache** is what keeps the hot path off the database. In real Lambda this cache
  lives in every Front-End Invoker with a TTL of ~60 seconds.

## 🌐 HTTP API (the contract)

| Method | Path | Purpose | Type |
|---|---|---|---|
| `POST` | `/v1/functions` | Create function (returns presigned S3 URL). | Control |
| `PUT`  | `/v1/functions/{name}/config` | Update env vars, memory, timeout. | Control |
| `POST` | `/v1/functions/{name}/versions` | Publish an immutable version. | Control |
| `POST` | `/v1/functions/{name}/invocations` (`X-Invocation-Type: RequestResponse`) | Sync invoke. | Data |
| `POST` | `/v1/functions/{name}/invocations` (`X-Invocation-Type: Event`) | Async invoke. | Data |
| `GET`  | `/v1/invocations/{id}` | Fetch logs & status. | Control |

**Why the split path?** Control-plane (register, configure) can be 100 QPS, strongly consistent.
Data-plane (invoke) can be 600 k QPS, eventually consistent. Running them on separate fleets
means a storm of deploys can't melt the execution path, and vice versa.

Next stop: the **hot path** itself — cold starts, warm pools, throttling, async retries.